In [1]:
# Load data into SQLite
import pandas as pd
import sqlite3

In [2]:
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# fixing TotalCharges (same cleaning as before)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges'])

# load into SQLite
conn = sqlite3.connect('../data/churn.db')
df.to_sql('customers', conn, if_exists='replace', index=False)
print(f"Database ready — {len(df)} rows loaded")


Database ready — 7032 rows loaded


In [3]:
def run_sql(query):
    result = pd.read_sql_query(query, conn)
    return result
    

In [4]:
#Query 1: Overall churn rate

In [5]:
run_sql("""
    SELECT 
        COUNT(*) AS total_customers,
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned,
        ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1.0 ELSE 0 END) / COUNT(*) * 100, 1) AS churn_rate_pct
    FROM customers
""")

,total_customers,churned,churn_rate_pct
0,7032,1869,26.6


In [6]:
#Query 2: Churn by contract type

In [7]:
q2 = run_sql("""
     Select 
          CONTRACT,
          COUNT(*) AS total_customers,
          SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned,
          ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1.0 ELSE 0 END) / COUNT(*) * 100, 1) AS churn_rate_pct
          FROM customers
          GROUP BY Contract
          ORDER BY churn_rate_pct DESC
          """)
q2

,Contract,total_customers,churned,churn_rate_pct
0,Month-to-month,3875,1655,42.7
1,One year,1472,166,11.3
2,Two year,1685,48,2.8


In [8]:
#Query 3: Churn by tenure bucket

In [9]:
q3 = run_sql("""
    SELECT 
        CASE 
            WHEN tenure BETWEEN 0 AND 6   THEN '0-6 mo'
            WHEN tenure BETWEEN 7 AND 12  THEN '7-12 mo'
            WHEN tenure BETWEEN 13 AND 24 THEN '13-24 mo'
            WHEN tenure BETWEEN 25 AND 48 THEN '25-48 mo'
            ELSE '49-72 mo'
        END AS tenure_bucket,
        COUNT(*) AS total_customers,
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned,
        ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1.0 ELSE 0 END) / COUNT(*) * 100, 1) AS churn_rate_pct
    FROM customers
    GROUP BY tenure_bucket
    ORDER BY churn_rate_pct DESC
""")
q3

,tenure_bucket,total_customers,churned,churn_rate_pct
0,0-6 mo,1470,784,53.3
1,7-12 mo,705,253,35.9
2,13-24 mo,1024,294,28.7
3,25-48 mo,1594,325,20.4
4,49-72 mo,2239,213,9.5


In [10]:
#Query 4: High risk segment

In [11]:
q4 = run_sql("""
    SELECT 
        Contract,
        CASE 
            WHEN tenure BETWEEN 0 AND 6   THEN '0-6 mo'
            WHEN tenure BETWEEN 7 AND 12  THEN '7-12 mo'
            ELSE '13+ mo'
        END AS tenure_bucket,
        COUNT(*) AS total_customers,
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned,
        ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1.0 ELSE 0 END) / COUNT(*) * 100, 1) AS churn_rate_pct
    FROM customers
    WHERE Contract = 'Month-to-month'
    GROUP BY tenure_bucket
    ORDER BY churn_rate_pct DESC
""")
q4

,Contract,tenure_bucket,total_customers,churned,churn_rate_pct
0,Month-to-month,0-6 mo,1413,780,55.2
1,Month-to-month,7-12 mo,581,244,42.0
2,Month-to-month,13+ mo,1881,631,33.5


In [12]:
# The above results concludes :
# - Month-to-month customers in their first 6 months churn at 55.2% — more than 1 in 2 customers.